# 2.2 — Séparer apprentissage, réglage et évaluation

| Partition | Rôle |
|---|---|
| TRAIN | Entraîner le modèle et construire le dictionnaire |
| VALIDATION | Choisir les réglages |
| TEST | Évaluation finale, sans aucun réglage |

1. Garder les partitions officielles, vérifier les doublons, sauvegarder les identifiants.
2. Fixer une convention BIO, la même pour les deux méthodes.

Sorties : `data/partitions_extraction.csv`, `data/corpus_extraction.jsonl` et `data/bio_extraction.jsonl` (utilisées en 2.3 et 2.4).

In [1]:
import sys, json, random
from collections import Counter
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.cdr import CATEGORIES, FICHIERS, charger_cdr, decouper_phrases, sauvegarder_jsonl

DOSSIER_CDR = RACINE / "CDR_Data" / "CDR.Corpus.v010516"
DATA = RACINE / "data"
RESULTATS = RACINE / "resultats" / "extraction"
SEED = 42

from tals import Mention, tokeniser_avec_offsets, mentions_vers_bio, bio_vers_mentions

corpus = charger_cdr(DOSSIER_CDR)

## Partitions officielles et doublons

BC5CDR fournit trois partitions de 500 résumés : `CDR_TrainingSet` → TRAIN, `CDR_DevelopmentSet` → VALIDATION, `CDR_TestSet` → TEST. Un résumé reste entier dans une seule partition. On vérifie qu'aucun document n'apparaît deux fois.

In [2]:
pmids = {nom: {d.document_id for d in docs} for nom, docs in corpus.items()}
textes = {nom: {d.texte for d in docs} for nom, docs in corpus.items()}
noms = list(corpus)
for i, a in enumerate(noms):
    for b in noms[i + 1:]:
        print(f"{a} ∩ {b} : {len(pmids[a] & pmids[b])} PMID communs, "
              f"{len(textes[a] & textes[b])} textes identiques")
for nom, docs in corpus.items():
    print(f"Doublons internes à {nom} : {len(docs) - len({d.texte for d in docs})}")

train ∩ validation : 0 PMID communs, 0 textes identiques
train ∩ test : 0 PMID communs, 0 textes identiques
validation ∩ test : 0 PMID communs, 0 textes identiques
Doublons internes à train : 0
Doublons internes à validation : 0
Doublons internes à test : 0


## Sauvegarde des identifiants de chaque partition

In [3]:
lignes = [{"document_id": d.document_id, "partition": nom}
          for nom, docs in corpus.items() for d in docs]
pd.DataFrame(lignes).to_csv(DATA / "partitions_extraction.csv", index=False)

tous = [d for docs in corpus.values() for d in docs]
sauvegarder_jsonl(tous, DATA / "corpus_extraction.jsonl")
print("Écrit :", DATA / "partitions_extraction.csv", "et", DATA / "corpus_extraction.jsonl")
pd.DataFrame(lignes).groupby("partition").size()

Écrit : /home/yanis/ESIEE_BIO/NLP/projet_nlp_sante/data/partitions_extraction.csv et /home/yanis/ESIEE_BIO/NLP/projet_nlp_sante/data/corpus_extraction.jsonl


partition
test          500
train         500
validation    500
dtype: int64

## Convention BIO

Chaque token reçoit une étiquette : `B-<catégorie>` (début de mention), `I-<catégorie>` (suite de la mention) ou `O` (hors mention). Les tokens sont les mots et les signes de ponctuation, séparés (`\w+|[^\w\s]`).

- **Mentions imbriquées** : on vérifie qu'il n'y en a pas.
- **Mentions composites** (« ovarian and testicular cancer ») : gardées telles qu'annotées, comme une seule mention.

Vérification : on retransforme les étiquettes BIO en mentions et on contrôle qu'on retrouve exactement les mentions d'origine.

In [4]:
bilan, mal_alignees, bio_lignes = [], [], []
for nom, docs in corpus.items():
    n = chevauchements = perdues = ajoutees = 0
    for d in docs:
        ref = {Mention(d.document_id, m.debut, m.fin, m.type) for m in d.mentions}
        chevauchements += sum(1 for a, b in zip(d.mentions, d.mentions[1:]) if b.debut < a.fin)
        tokens = tokeniser_avec_offsets(d.texte)
        offsets = [(a, b) for _, a, b in tokens]
        debuts, fins = {a for a, _ in offsets}, {b for _, b in offsets}
        for m in sorted(ref):
            if m.debut not in debuts or m.fin not in fins:
                mal_alignees.append((nom, d, m))
        tags = mentions_vers_bio(ref, offsets)
        retour = bio_vers_mentions(tags, offsets, d.document_id, politique="stricte")
        n += len(ref); perdues += len(ref - retour); ajoutees += len(retour - ref)
        bio_lignes.append({"document_id": d.document_id, "partition": nom,
                           "tokens": [s for s, _, _ in tokens], "offsets": offsets, "tags": tags})
    bilan.append({"partition": nom, "mentions": n, "chevauchements": chevauchements,
                  "composites": sum(m.composite for d in docs for m in d.mentions),
                  "bornes hors frontière de token": sum(1 for p, *_ in mal_alignees if p == nom),
                  "perdues à l'aller-retour BIO": perdues, "créées à l'aller-retour BIO": ajoutees})
bilan = pd.DataFrame(bilan).set_index("partition")
bilan.to_csv(RESULTATS / "bilan_bio.csv")
bilan

,mentions,chevauchements,composites,bornes hors frontière de token,perdues à l'aller-retour BIO,créées à l'aller-retour BIO
partition,,,,,,
train,9385,0,107,0,0,0
validation,9591,0,89,10,10,6
test,9809,0,57,5,5,1


### Mentions qui commencent ou finissent au milieu d'un token

Exemple : « 5-HT » dans « 5-HT6 ». Le BIO ne peut pas les représenter exactement, donc aucune des deux méthodes ne pourra les retrouver. Elles restent dans la référence et comptent comme erreurs.

In [5]:
pd.DataFrame([{"partition": p, "PMID": d.document_id, "mention": d.texte[m.debut:m.fin], "type": m.type,
               "contexte": d.texte[max(0, m.debut - 15):m.debut] + "⟦" + d.texte[m.debut:m.fin] + "⟧"
                           + d.texte[m.fin:m.fin + 15]}
              for p, d, m in mal_alignees])

,partition,PMID,mention,type,contexte
0,validation,20705401,5-HT,Chemical,e serotonin 6 (⟦5-HT⟧6) receptor is
1,validation,20705401,5-HT,Chemical,of a selective ⟦5-HT⟧6 receptor anta
2,validation,20705401,5-HT,Chemical,as restored by ⟦5-HT⟧6 receptor anta
3,validation,20705401,5-HT,Chemical,s that altered ⟦5-HT⟧6 receptors are
4,validation,20705401,5-HT,Chemical,ciation of the ⟦5-HT⟧6 gene (HTR6) w
5,validation,19692487,Cl,Chemical,e of the Na-K-2⟦Cl⟧ cotransporter
6,validation,15075188,Cl,Chemical,"), Na(+)-K(+)-2⟦Cl⟧(-) cotransport"
7,validation,2907585,5-hydroxytryptamine,Chemical,g affinity for ⟦5-hydroxytryptamine⟧1a recepto...
8,validation,1969772,DA,Chemical,"c dopamine-1, (⟦DA⟧1) and dopamine"
9,validation,1969772,DA,Chemical,nd dopamine-2 (⟦DA⟧2) receptor ago


### Exemple de séquence BIO

In [6]:
exemple = next(l for l in bio_lignes if l["document_id"] == "227508")
pd.DataFrame({"token": exemple["tokens"][:30], "offsets": exemple["offsets"][:30],
              "étiquette": exemple["tags"][:30]}).T

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
token,Naloxone,reverses,the,antihypertensive,effect,of,clonidine,.,In,unanesthetized,...,heart,rate,produced,by,intravenous,clonidine,",",5,to,20
offsets,"(0, 8)","(9, 17)","(18, 21)","(22, 38)","(39, 45)","(46, 48)","(49, 58)","(58, 59)","(60, 62)","(63, 77)",...,"(146, 151)","(152, 156)","(157, 165)","(166, 168)","(169, 180)","(181, 190)","(190, 191)","(192, 193)","(194, 196)","(197, 199)"
étiquette,B-Chemical,O,O,O,O,O,B-Chemical,O,O,O,...,O,O,O,O,O,B-Chemical,O,O,O,O


### Mentions coupées par une fin de phrase

Le modèle du 2.4 traite le texte par groupes de phrases. Si une fin de phrase est détectée au milieu d'une mention, cette mention est perdue : on les compte. Le 2.4 regroupe donc des phrases entières, puis remet les mentions à leur position dans le document.

In [1]:
coupees = []
for nom, docs in corpus.items():
    for d in docs:
        phrases = decouper_phrases(d.texte)
        for m in d.mentions:
            if not any(a <= m.debut and m.fin <= b for a, b in phrases):
                coupees.append({"partition": nom, "PMID": d.document_id, "mention": m.texte, "type": m.type})
print(f"Mentions coupées par une frontière de phrase : {len(coupees)}")
pd.DataFrame(coupees)

NameError: name 'corpus' is not defined

In [10]:
with open(DATA / "bio_extraction.jsonl", "w", encoding="utf-8") as f:
    for l in bio_lignes:
        f.write(json.dumps(l, ensure_ascii=False) + "\n")
print("Écrit :", DATA / "bio_extraction.jsonl", f"({len(bio_lignes)} documents)")

Écrit : /home/yanis/ESIEE_BIO/NLP/projet_nlp_sante/data/bio_extraction.jsonl (1500 documents)


## Bilan des conventions (identiques pour les deux méthodes)

- Partitions officielles, sans doublon ; identifiants dans `data/partitions_extraction.csv`.
- Positions en caractères `[debut, fin[` sur `titre + " " + résumé`.
- Pas de mention imbriquée ; mentions composites gardées entières.
- BIO exact, sauf pour les mentions qui coupent un token (aucune dans TRAIN).